<a href="https://colab.research.google.com/github/farezae/mscproj/blob/main/AMT_tokenisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt install fluidsynth

!git clone https://github.com/jthickstun/anticipation.git
!pip install ./anticipation
!pip install -r anticipation/requirements.txt
!pip install matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fluid-soundfont-gm libevdev2 libfluidsynth3 libgudev-1.0-0 libinput-bin libinput10
  libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5
  libqt5svg5 libqt5widgets5 libwacom-bin libwacom-common libwacom9 libxcb-icccm4 libxcb-image0
  libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1
  libxkbcommon-x11-0 qsynth qt5-gtk-platformtheme qttranslations5-l10n timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  fluid-soundfont-gm fluidsynth libevdev2 libfluidsynth3 libgudev-1.0-0 libinput-bin libinput10
  libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5
  libqt5svg5 libqt5widgets5 libwacom-bin libwacom-common libwacom9 libxcb-icc

In [ ]:
import sys,time
import numpy

import midi2audio
import transformers
import os

from transformers import AutoModelForCausalLM
from pathlib import Path
from IPython.display import Audio

from anticipation import ops
from anticipation.sample import generate
from anticipation.tokenize import extract_instruments
from anticipation.convert import events_to_midi,midi_to_events
from anticipation.config import *
from anticipation.vocab import *

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# download latest version of clean lakh dataset
path = kagglehub.dataset_download("imsparsh/lakh-midi-clean")
midi_paths=Path(path)
print("Path to dataset files:", midi_paths)

# list all .mid files
midi_files = list(midi_paths.rglob("*.mid"))
for file in midi_files:
    print(file)

100%|██████████| 226M/226M [00:11<00:00, 20.3MB/s]

Extracting files...


Streaming output truncated to the last 5000 lines.
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Ferite_e_lacrime_You_.1.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Tutti_quanti_abbiamo_un_angelo.1.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Mi_hai_preso_il_cuore.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Una_Citta_Per_Cantare.1.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Vorrei_incontrarti_fra_centanni_con_la_partecipazione_di_Tosca_.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Il_sole_e_la_luna.1.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Non_abbiam_bisogno_di_parole.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Una_Citta_Per_Cantare.mid
/root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Vorrei_incontrarti_fra_centanni

In [ ]:
SMALL_MODEL = 'stanford-crfm/music-small-800k'     # faster inference, worse sample quality
MEDIUM_MODEL = 'stanford-crfm/music-medium-800k'   # slower inference, better sample quality
LARGE_MODEL = 'stanford-crfm/music-large-800k'     # slowest inference, best sample quality

# load an anticipatory music transformer
model = AutoModelForCausalLM.from_pretrained(SMALL_MODEL).cuda()

# a MIDI synthesizer
fs = midi2audio.FluidSynth('/usr/share/sounds/sf2/FluidR3_GM.sf2')

# the MIDI synthesis script
def synthesize(fs, tokens):
    mid = events_to_midi(tokens)
    mid.save('tmp.mid')
    fs.midi_to_audio('tmp.mid', 'tmp.wav')
    return 'tmp.wav'

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at stanford-crfm/music-small-800k were not used when initializing GPT2LMHeadModel: ['token_out_embeddings']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# setup folder to store tokenisations in drive
output_folder_path = '/content/drive/MyDrive/MusicData/lakh_generated_events'
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)
    print(f"Created output folder: {output_folder_path}")

In [ ]:
for filepath in midi_files:
  try:
    # convert MIDI to events
    events = midi_to_events(str(filepath))

    # each tokenisation needs a unique filename
    base_name = os.path.splitext(filepath)[0] # get file name without extension
    directory, file_with_extension = os.path.split(base_name)
    output_file_path= os.path.join(output_folder_path, f"{file_with_extension}_events.txt")

    # write the events into a text file
    with open(output_file_path, 'w') as f:
      for item in range(0,len(events)):
        f.write(str(events[item]) + '\n')
    print(f"Successfully processed and saved events for: {filepath}") # confirm writing has been successful

  except:
    print(f"Error processing {filepath}")


Streaming output truncated to the last 5000 lines.
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Ferite_e_lacrime_You_.1.mid
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Tutti_quanti_abbiamo_un_angelo.1.mid
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Mi_hai_preso_il_cuore.mid
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Una_Citta_Per_Cantare.1.mid
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Vorrei_incontrarti_fra_centanni_con_la_partecipazione_di_Tosca_.mid
Successfully processed and saved events for: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1/Ron/Il_sole_e_la_luna.1.mid
Successfully processed and saved eve

In [ ]:
# find out how many of the files were processed - issue, 1493 files have not been tokenised.

if os.path.exists(output_folder_path):
    file_count = len([name for name in os.listdir(output_folder_path) if os.path.isfile(os.path.join(output_folder_path, name))])
    print(f"Number of processed files uploaded: {file_count}")
else:
    print(f"The directory {output_folder_path} does not exist.")

print(f"Number of files in the lakh dataset: {len(midi_files)}")


Number of processed files uploaded: 15739
Number of files in the lakh dataset: 17232
